In [ ]:
```xml
<VSCode.Cell language="markdown">
# ⚡ Reactive Programming & Backpressure Guide

**Building responsive, resilient systems with reactive streams and backpressure handling**

This guide covers:
- Reactive programming principles
- Observable and stream patterns
- Backpressure strategies
- Operators and transformations
- Error handling in reactive systems
- Concurrency and scheduling
- Testing reactive code
- Real-world reactive patterns
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Reactive Streams & Observables

### Observable Pattern
```python
from typing import Callable, List, Optional, Dict, Any
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
import threading

class StreamEvent(Enum):
    """Stream event types"""
    NEXT = "next"
    ERROR = "error"
    COMPLETE = "complete"

@dataclass
class StreamItem:
    """Item in stream"""
    value: Any
    timestamp: datetime
    source: str

class Observer:
    """Observer pattern for streams"""
    
    def __init__(self):
        self.on_next: Optional[Callable] = None
        self.on_error: Optional[Callable] = None
        self.on_complete: Optional[Callable] = None
    
    def next(self, value: Any):
        """Emit next value"""
        if self.on_next:
            self.on_next(value)
    
    def error(self, error: Exception):
        """Emit error"""
        if self.on_error:
            self.on_error(error)
    
    def complete(self):
        """Signal completion"""
        if self.on_complete:
            self.on_complete()

class Observable:
    """Observable stream"""
    
    def __init__(self, source: Callable):
        self.source = source
        self.observers: List[Observer] = []
        self.subscriptions: List[Dict] = []
    
    def subscribe(self, observer: Observer) -> 'Subscription':
        """Subscribe observer"""
        
        self.observers.append(observer)
        
        subscription = Subscription(self, observer)
        self.subscriptions.append(subscription)
        
        # Execute source with observer
        try:
            self.source(observer)
        except Exception as e:
            observer.error(e)
        
        return subscription
    
    def map(self, transform: Callable) -> 'Observable':
        """Transform values"""
        
        def new_source(observer: Observer):
            mapped_observer = Observer()
            
            def on_next_mapped(value):
                try:
                    transformed = transform(value)
                    observer.next(transformed)
                except Exception as e:
                    observer.error(e)
            
            mapped_observer.on_next = on_next_mapped
            mapped_observer.on_error = observer.error
            mapped_observer.on_complete = observer.complete
            
            self.subscribe(mapped_observer)
        
        return Observable(new_source)
    
    def filter(self, predicate: Callable) -> 'Observable':
        """Filter values"""
        
        def new_source(observer: Observer):
            filtered_observer = Observer()
            
            def on_next_filtered(value):
                if predicate(value):
                    observer.next(value)
            
            filtered_observer.on_next = on_next_filtered
            filtered_observer.on_error = observer.error
            filtered_observer.on_complete = observer.complete
            
            self.subscribe(filtered_observer)
        
        return Observable(new_source)
    
    def take(self, n: int) -> 'Observable':
        """Take first n items"""
        
        def new_source(observer: Observer):
            count = [0]  # Mutable counter
            
            take_observer = Observer()
            
            def on_next_take(value):
                if count[0] < n:
                    observer.next(value)
                    count[0] += 1
                    
                    if count[0] >= n:
                        observer.complete()
            
            take_observer.on_next = on_next_take
            take_observer.on_error = observer.error
            take_observer.on_complete = observer.complete
            
            self.subscribe(take_observer)
        
        return Observable(new_source)

class Subscription:
    """Represents subscription to observable"""
    
    def __init__(self, observable: Observable, observer: Observer):
        self.observable = observable
        self.observer = observer
        self.disposed = False
    
    def unsubscribe(self):
        """Unsubscribe from observable"""
        if not self.disposed:
            if self.observer in self.observable.observers:
                self.observable.observers.remove(self.observer)
            self.disposed = True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Backpressure Handling

### Backpressure Strategy
```python
class BackpressureStrategy(Enum):
    """Backpressure handling strategies"""
    BUFFER = "buffer"
    DROP = "drop"
    ERROR = "error"
    IGNORE = "ignore"

class BackpressureHandler:
    """Handle backpressure in streams"""
    
    def __init__(self, strategy: BackpressureStrategy = BackpressureStrategy.BUFFER,
                 buffer_size: int = 100):
        self.strategy = strategy
        self.buffer_size = buffer_size
        self.buffer: List[Any] = []
        self.dropped_count = 0
        self.errors_count = 0
    
    def handle_item(self, item: Any) -> bool:
        """Handle incoming item with backpressure"""
        
        if self.strategy == BackpressureStrategy.BUFFER:
            if len(self.buffer) < self.buffer_size:
                self.buffer.append(item)
                return True
            else:
                return False  # Buffer full
        
        elif self.strategy == BackpressureStrategy.DROP:
            if len(self.buffer) < self.buffer_size:
                self.buffer.append(item)
                return True
            else:
                self.dropped_count += 1
                return False
        
        elif self.strategy == BackpressureStrategy.ERROR:
            if len(self.buffer) < self.buffer_size:
                self.buffer.append(item)
                return True
            else:
                self.errors_count += 1
                raise OverflowError("Buffer overflow - backpressure triggered")
        
        else:  # IGNORE
            self.buffer.append(item)
            return True
    
    def get_buffer_status(self) -> Dict:
        """Get buffer status"""
        return {
            'buffer_size': len(self.buffer),
            'max_size': self.buffer_size,
            'utilization_percent': (len(self.buffer) / self.buffer_size) * 100,
            'dropped_items': self.dropped_count,
            'errors': self.errors_count
        }
    
    def drain_buffer(self) -> List[Any]:
        """Drain buffer"""
        items = self.buffer.copy()
        self.buffer.clear()
        return items

class FlowController:
    """Control flow of items in stream"""
    
    def __init__(self, capacity: int = 100):
        self.capacity = capacity
        self.pending_items = 0
        self.requested_items = 0
    
    def request(self, n: int) -> int:
        """Request n items"""
        self.requested_items = n
        return n
    
    def offer(self, item: Any) -> bool:
        """Offer item if capacity available"""
        if self.pending_items < self.capacity:
            self.pending_items += 1
            self.requested_items = max(0, self.requested_items - 1)
            return True
        
        return False
    
    def can_accept_more(self) -> bool:
        """Check if can accept more items"""
        return self.pending_items < self.capacity and self.requested_items > 0
    
    def get_flow_status(self) -> Dict:
        """Get flow control status"""
        return {
            'capacity': self.capacity,
            'pending': self.pending_items,
            'requested': self.requested_items,
            'available_slots': self.capacity - self.pending_items
        }
```

### Rate Limiting in Streams
```python
class RateLimiter:
    """Rate limit stream items"""
    
    def __init__(self, items_per_second: float):
        self.items_per_second = items_per_second
        self.last_item_time: Optional[datetime] = None
        self.item_count = 0
        self.throttled_count = 0
    
    def should_emit(self) -> bool:
        """Check if item should be emitted"""
        
        now = datetime.now()
        
        if not self.last_item_time:
            self.last_item_time = now
            return True
        
        time_elapsed = (now - self.last_item_time).total_seconds()
        items_allowed = time_elapsed * self.items_per_second
        
        if self.item_count < items_allowed:
            self.item_count += 1
            return True
        else:
            self.throttled_count += 1
            return False
    
    def get_rate_status(self) -> Dict:
        """Get rate limiting status"""
        return {
            'items_per_second': self.items_per_second,
            'emitted': self.item_count,
            'throttled': self.throttled_count,
            'throttle_rate': (self.throttled_count / (self.item_count + self.throttled_count)) if (self.item_count + self.throttled_count) > 0 else 0
        }

class AdaptiveRateLimiter:
    """Adapt rate based on downstream capacity"""
    
    def __init__(self, initial_rate: float = 10.0):
        self.current_rate = initial_rate
        self.min_rate = 1.0
        self.max_rate = 1000.0
    
    def adjust_for_backpressure(self, backpressure_detected: bool):
        """Adjust rate based on backpressure"""
        
        if backpressure_detected:
            # Reduce rate
            self.current_rate = max(self.current_rate * 0.8, self.min_rate)
        else:
            # Increase rate
            self.current_rate = min(self.current_rate * 1.1, self.max_rate)
    
    def get_current_rate(self) -> float:
        """Get current rate"""
        return self.current_rate
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Error Handling & Resilience

### Error Handling in Streams
```python
class ErrorRecoveryStrategy(Enum):
    """Error recovery strategies"""
    RETRY = "retry"
    FALLBACK = "fallback"
    PROPAGATE = "propagate"

class ResilientObservable(Observable):
    """Observable with error recovery"""
    
    def __init__(self, source: Callable):
        super().__init__(source)
        self.error_handlers: List[Callable] = []
    
    def retry(self, max_retries: int = 3, backoff_base: float = 2.0) -> 'Observable':
        """Retry on error"""
        
        def new_source(observer: Observer):
            attempt = [0]
            
            def subscribe_with_retry():
                try:
                    self.source(observer)
                except Exception as e:
                    if attempt[0] < max_retries:
                        attempt[0] += 1
                        backoff = (backoff_base ** attempt[0])
                        print(f"Retrying (attempt {attempt[0]}), backoff: {backoff}s")
                        
                        # In real system, would schedule retry
                        subscribe_with_retry()
                    else:
                        observer.error(e)
            
            subscribe_with_retry()
        
        return Observable(new_source)
    
    def catch_error(self, fallback_source: Callable) -> 'Observable':
        """Use fallback on error"""
        
        def new_source(observer: Observer):
            error_observer = Observer()
            
            def on_error_fallback(error: Exception):
                # Switch to fallback
                fallback_source(observer)
            
            error_observer.on_next = observer.next
            error_observer.on_error = on_error_fallback
            error_observer.on_complete = observer.complete
            
            self.subscribe(error_observer)
        
        return Observable(new_source)
    
    def timeout(self, timeout_seconds: float) -> 'Observable':
        """Add timeout to stream"""
        
        def new_source(observer: Observer):
            timer_expired = [False]
            
            def timeout_handler():
                timer_expired[0] = True
                observer.error(TimeoutError(f"Stream timeout after {timeout_seconds}s"))
            
            # In real system, would schedule timeout
            self.subscribe(observer)
        
        return Observable(new_source)
```

### Resilience Metrics
```python
class ResilienceMetrics:
    """Track resilience metrics"""
    
    def __init__(self):
        self.errors: List[Dict] = []
        self.recoveries: List[Dict] = []
        self.timeouts: int = 0
        self.retries: int = 0
    
    def record_error(self, error: Exception, context: Dict = None):
        """Record error"""
        self.errors.append({
            'error': str(error),
            'type': type(error).__name__,
            'timestamp': datetime.now(),
            'context': context or {}
        })
    
    def record_recovery(self, method: str):
        """Record successful recovery"""
        self.recoveries.append({
            'method': method,
            'timestamp': datetime.now()
        })
    
    def get_resilience_score(self) -> float:
        """Calculate resilience score (0-1)"""
        
        total_events = len(self.errors) + len(self.recoveries)
        
        if total_events == 0:
            return 1.0
        
        recovery_rate = len(self.recoveries) / total_events
        
        return recovery_rate
    
    def get_metrics_summary(self) -> Dict:
        """Get metrics summary"""
        
        return {
            'total_errors': len(self.errors),
            'total_recoveries': len(self.recoveries),
            'timeouts': self.timeouts,
            'retries': self.retries,
            'resilience_score': self.get_resilience_score(),
            'error_types': list(set(e['type'] for e in self.errors))
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Reactive Programming & Backpressure Checklist

✅ **Observable & Streams**
- [ ] Observable pattern implemented
- [ ] Operators available (map, filter, etc.)
- [ ] Subscriptions managed
- [ ] Unsubscription handling
- [ ] Memory leaks prevented

✅ **Backpressure Handling**
- [ ] Backpressure strategy chosen
- [ ] Buffer sizes configured
- [ ] Flow control implemented
- [ ] Buffer monitoring active
- [ ] Overflow handling

✅ **Error Handling**
- [ ] Error handlers registered
- [ ] Retry logic implemented
- [ ] Fallback strategies
- [ ] Timeout handling
- [ ] Error propagation clean

✅ **Performance & Resilience**
- [ ] Rate limiting applied
- [ ] Adaptive rate limiting
- [ ] Metrics tracked
- [ ] Recovery mechanisms
- [ ] Throughput optimized

✅ **Operational**
- [ ] Monitoring alerts
- [ ] Backpressure detected
- [ ] Metrics dashboard
- [ ] Load testing done
- [ ] Documentation complete
</VSCode.Cell>
```